# Rearrangement experiment run - preload + trigger

One full cycle using real fluorescence images in place of the camera: find trap
sites from the averaged image set, take one image as the captured occupancy
shot, then generate and deliver the sequence via preload + hardware trigger.

**To go live**: point `IMAGE_DIR` at a one-off calibration capture (or keep
using a saved set) for the sites/threshold cell, and replace the
`captured_image = images[0]` line with `camera.acquire_n_frames(1)[0]`.
Everything downstream - occupancy, sequence generation, delivery - is
unchanged.

In [3]:
from pathlib import Path

import numpy as np
from PIL import Image

import pytweezer.phasemask as pm
from pytweezer.analysis import analysis as an
from pytweezer.arduino import ArduinoPulser
from pytweezer.coordinators.rearrangement import Rearrangement
from pytweezer.drivers.slm import SLM, SimulatedSLM

In [4]:
def find_repo_root(marker="pyproject.toml"):
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"no {marker} found above {Path.cwd()}")


IMAGE_DIR = find_repo_root() / "Local_Files" / "Image Data" / "100ms expo"
GRID_SHAPE = (16, 16)
WINDOW_SIZE = 3

D0 = 1.2
PROFILE = "linear"
PULSE_PERIOD_US = 700
ARDUINO_PORT = "COM5"
USE_REAL_SLM = True

In [5]:
dx, dy = 8.0, 4.0
PM = pm.OptimisationBasedPhasemaskGeneratorGPU(
    wavelength_um=0.852,
    focal_length_mm=17.3,
    slm_pitch_um=17,
    slm_res=(1024, 1024),
    input_beam_waist_mm=16,
    fresnel_f_mm=1072,
    blaze_dx_dy_um=(40 + dx, -8 + dy),
    zernike_coeff_dict={
        5: 1.195,
        6: 0.725,
        7: 0.970,
        8: 0.478,
        9: -1.091,
        10: 0.303,
        11: 0.021,
        12: 0.072,
        13: 0.049,
    },
)

spacing_um, angle_deg, wgs_iters = 5.0, 2.5, 1000
target_init = PM.generate_weighted_array(
    np.ones(GRID_SHAPE), spacing_um, init_phase_randomness=1.0, angle_deg=angle_deg
)
_, terms_init, _ = PM.superposition_optimization(
    target_init, max_iter=wgs_iters, damping=0.5, verbose=False
)
target_final = PM.generate_weighted_array(
    np.ones((10, 10)), spacing_um, init_phase_randomness=1.0, angle_deg=angle_deg
)
_, terms_final, _ = PM.superposition_optimization(
    target_final, max_iter=wgs_iters, damping=0.5, verbose=False
)

--- System Configuration ---
SLM Plane Width: 17.41 mm
SLM Plane Height: 17.41 mm
Focal Plane Resolution x (pixel size): 0.8467 um
Focal Plane Resolution y (pixel size): 0.8467 um
Focal Plane Width: 867.04 um
Focal Plane Height: 867.04 um
Fresnel Lens Focal Length: 1072.00 mm
Blazed Grating Displacement (dx, dy): (48.0, -4.0) um
--- Target Generation ---
Grid: 16x16
Spacing: 5.0 um
--- Starting GPU Superposition Phase Retrieval ---
Iteration 999 | Mean-Squared Error: 1.53e-13 | Uniformity: 99.99% | Min/Max ratio: 0.999
Optimization finished in 3.65 seconds.
--- Target Generation ---
Grid: 10x10
Spacing: 5.0 um
--- Starting GPU Superposition Phase Retrieval ---
Iteration 999 | Mean-Squared Error: 2.09e-12 | Uniformity: 99.99% | Min/Max ratio: 0.999
Optimization finished in 1.92 seconds.


## Calibration: trap sites + occupancy threshold

One-off, from the averaged image set - not repeated per shot. Site-finding needs
the tophat background removed first or it never converges on an exact count.

In [6]:
paths = sorted(IMAGE_DIR.glob("*.tiff"), key=lambda p: int(p.stem.split("_")[1]))
images = np.stack([np.array(Image.open(p).convert("L")) for p in paths]).astype(
    np.float64
)

mean_th = an.morphological_tophat_high_pass(images.mean(axis=0), feature_size=10)
grid_positions, _ = an.detect_trap_sites(mean_th, GRID_SHAPE, detection_step=1)

site_sums = np.array(
    [
        an.sum_pixel_values(
            an.morphological_tophat_high_pass(img, feature_size=10),
            grid_positions,
            GRID_SHAPE,
            window_size=WINDOW_SIZE,
        ).flatten()
        for img in images
    ]
)
threshold, *_ = an.detect_loading_threshold(site_sums.flatten())
print(
    f"{len(grid_positions)} sites, threshold={threshold:.1f}, "
    f"{(site_sums > threshold).sum(axis=1).mean():.0f}/{GRID_SHAPE[0] * GRID_SHAPE[1]} atoms/shot mean"
)

Looking for trap sites...
16x16 array detected.
256 sites, threshold=275.0, 135/256 atoms/shot mean


## Hardware

In [7]:
if USE_REAL_SLM:
    try:
        slm = SLM(wait_for_trigger=False, output_pulse=True)
        print(f"SLM: real board {slm.get_dimensions()}")
    except Exception as exc:
        slm = SimulatedSLM()
        print(f"SLM: SimulatedSLM  <-- real driver FAILED: {type(exc).__name__}: {exc}")
else:
    slm = SimulatedSLM()
    print("SLM: SimulatedSLM - not a real run")

coordinator = Rearrangement({"slm": slm}, {})
coordinator._state = {"grid_positions": grid_positions}  # enough for _extract_occupancy

arduino = ArduinoPulser(ARDUINO_PORT)
print("Arduino:", arduino.connect())

SLM: SimulatedSLM  <-- real driver FAILED: RuntimeError: Blink SDK did not construct successfully
Arduino: Listening for Python...


## One experiment run: captured image -> occupancy -> deliver via preload + trigger

In [8]:
# use camera grab here instead
captured_image = images[0]

occ_mask = coordinator._extract_occupancy(captured_image, GRID_SHAPE, threshold)
print(f"occupancy: {occ_mask.sum()} / {occ_mask.size} atoms loaded")

slm.set_wait_for_trigger(False)
frames = PM.iter_rearrangement_sequence(
    terms_init, terms_final, occ_mask, d0=D0, profile=PROFILE, to_host=False
)
n = coordinator._preload_sequence_pipelined(frames)

slm.set_wait_for_trigger(True)
slm.start_auto_increment(n)
span = arduino.send_pulses(n - 1, period_us=PULSE_PERIOD_US)
slm.stop_auto_increment()
slm.set_wait_for_trigger(False)

period = span / max(n - 2, 1)
print(f"{n} frames delivered, move = {(n - 1) * period * 1e3:.2f} ms")

occupancy: 139 / 256 atoms loaded
13 frames delivered, move = 8.38 ms


## Close

In [9]:
arduino.close()
slm.close()
print("closed")

closed
